# Import Libraries

In [17]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# API function

In [2]:
def get_carbon_intensity(start_date, end_date):
    url = f"https://api.carbonintensity.org.uk/intensity/{start_date}/{end_date}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    return pd.json_normalize(data["data"])

# Date range
2025-01-01 to 2026-01-01

In [3]:
start = datetime(2025, 1, 1)
end = datetime(2026, 1, 1)

In [4]:
dfs = []
current = start

while current < end:
    next_date = min(current + timedelta(days=30), end)

    df = get_carbon_intensity(
        current.strftime("%Y-%m-%d"),
        next_date.strftime("%Y-%m-%d")
    )

    dfs.append(df)

    current = next_date

# Combine all chunks

In [5]:
df_ci = pd.concat(dfs, ignore_index=True)
df_ci.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low


In [6]:
df_ci.columns

Index(['from', 'to', 'intensity.forecast', 'intensity.actual',
       'intensity.index'],
      dtype='str')

# Clean the Dataset

In [7]:
# Rename dotted columns
df_ci = df_ci.rename(columns={
    "intensity.actual": "actual",
    "intensity.forecast": "forecast"
})

# Create datetime column from 'from'
df_ci["datetime"] = pd.to_datetime(df_ci["from"], utc=True)

# Remove timezone
df_ci["datetime"] = df_ci["datetime"].dt.tz_localize(None)

# Filter to 2025 only
df_ci = df_ci[
    (df_ci["datetime"] >= "2025-01-01") &
    (df_ci["datetime"] < "2026-01-01")
]

# Drop forecast column
df_ci = df_ci.drop(columns=["forecast"])

# Rename actual to carbon_intensity
df_ci = df_ci.rename(columns={"actual": "carbon_intensity"})

# Add units column
df_ci["units"] = "gCO2/kWh"

# Keep only required columns
df_ci = df_ci[["datetime", "carbon_intensity", "units"]]

# Sort and remove duplicate datetimes
df_ci = (
    df_ci
    .sort_values("datetime")
    .drop_duplicates(subset="datetime")
    .reset_index(drop=True)
)

df_ci.head()

,datetime,carbon_intensity,units
0,2025-01-01 00:00:00,55,gCO2/kWh
1,2025-01-01 00:30:00,54,gCO2/kWh
2,2025-01-01 01:00:00,53,gCO2/kWh
3,2025-01-01 01:30:00,53,gCO2/kWh
4,2025-01-01 02:00:00,47,gCO2/kWh


In [8]:
# Check missing values
df_ci.isnull().sum()

datetime            0
carbon_intensity    0
units               0
dtype: int64

In [9]:
# Check duplicates
df_ci.duplicated().sum()

np.int64(0)

# Datetime Frequency

In [10]:
df_ci["datetime"].diff().value_counts().head()

datetime
0 days 00:30:00    17519
Name: count, dtype: int64

In [11]:
# dataset shape
df_ci.shape

(17520, 3)

In [12]:
# date range
print("Start:", df_ci["datetime"].min())
print("End:", df_ci["datetime"].max())

Start: 2025-01-01 00:00:00
End: 2025-12-31 23:30:00


# Preview final cleaned dataset

In [16]:
df_ci.head()

,datetime,carbon_intensity,units
0,2025-01-01 00:00:00,55,gCO2/kWh
1,2025-01-01 00:30:00,54,gCO2/kWh
2,2025-01-01 01:00:00,53,gCO2/kWh
3,2025-01-01 01:30:00,53,gCO2/kWh
4,2025-01-01 02:00:00,47,gCO2/kWh


# Cleaned file

In [14]:
df_ci.to_csv("carbon_intensity_2025_lean.csv", index=False)